In [1]:
# ---- version 1.2 ------
# Read linkedin pdf
# Read summary document
# compile system prompt and concatenate the linkedin & summary details with it as context
# invoke openai api and send the user messages, the usual way
# create an evaluation function with gemini api
# capture response and print the evaluation
# use gradio as UI

In [2]:
import os
from dotenv import load_dotenv
from pydantic import BaseModel
from pypdf import PdfReader
from openai import OpenAI
import gradio as gr

In [3]:
reader = PdfReader("linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text
        
print(linkedin)

   
Contact
debmalyamondal63@gmail.com
www.linkedin.com/in/
debmalyamondal (LinkedIn)
medium.com/@debtalks (Blog)
Top Skills
Seaborn
Data Analysis
Problem Solving
Languages
English (Full Professional)
Hindi (Professional Working)
Bengali (Native or Bilingual)
Certifications
CFA Institute Investment Foundation
Data Science and AI Certificate for
Managers and Leaders
Text Prompt Engineering
Techniques Skill Badge
Microsoft Certified: Azure
Fundamentals
Honors-Awards
Winner - Trailblazer 2.0
Quarterly Excellence Award
Debmalya Mondal
Technical Business Analyst | AI/ML Enthusiast | Help organizations
to develop effective technology solutions
Bengaluru, Karnataka, India
Summary
Hi! Welcome to my profile!
This is Deb, born and brought up in Kolkata, currently staying in
Bangalore, India. I'm working as a technical business analyst/project
manager for one of the leading global banks. Throughout my career
I have played many roles from back-end development to business
analysis and project manag

In [4]:
with open("summary.txt", 'r', encoding='utf-8') as f:
    summary = f.read()
    
print(summary)

I am Debmalya Mondal. I am an IT business analyst, data scientist and AI consultant. I have predominatly worked in capital markets projects. In my 15
years of industry experience, I have played many roles like business analyst, project manager, and back-end java developer. I am originally from West Bengal, India
but stayed many places across the globe. I am a big time foodie and a really good home cook.


In [6]:
name = "Debmalya Mondal"

In [7]:
system_prompt = f"You are acting as {name}. You introduce yourself as {name}'s digital avatar. You are answering questions on behalf \
of {name} about {name}'s career, background, skills, and experience. Your responsibility to provide the website visitors with \
valid information and represent {name} as honest and truthful as possible. Be professional and engaging in your responses as you may \
be speaking to a potential employer or customer. If you don't know the answer say so. If the user does not ask questions about {name}'s \
career or skills, then humbly ask the user to stick to the topic and deny answering."

system_prompt += f"\n\n##Summary:\n{summary}\n\n"
system_prompt += f"\n\n##LinkedIn: \n{linkedin}\n\n"
system_prompt += "\n\nUsing this context, chat with the user"

In [8]:
print(system_prompt)

You are acting as Debmalya Mondal. You introduce yourself as Debmalya Mondal's digital avatar. You are answering questions on behalf of Debmalya Mondal about Debmalya Mondal's career, background, skills, and experience. Your responsibility to provide the website visitors with valid information and represent Debmalya Mondal as honest and truthful as possible. Be professional and engaging in your responses as you may be speaking to a potential employer or customer. If you don't know the answer say so. If the user does not ask questions about Debmalya Mondal's career or skills, then humbly ask the user to stick to the topic and deny answering.

##Summary:
I am Debmalya Mondal. I am an IT business analyst, data scientist and AI consultant. I have predominatly worked in capital markets projects. In my 15
years of industry experience, I have played many roles like business analyst, project manager, and back-end java developer. I am originally from West Bengal, India
but stayed many places ac

In [9]:
evaluation_system_prompt = f"You are an evaluator that decides the validity of a response to a question. You are provided with \
a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable. The Agent is \
playing a role of {name}'s digital avatar and is representing {name} on their website. The Agent is instructed to be professional \
and engaging since it will be chatting to a potential client or a future employer. The Agent is also instructed to answer only about \
{name}'s career, experience and skills and humbly refuse to answer any other query. The Agent has been provided with context \
on {name} in the form of summary and Linkedin details. Here's the information:"

evaluation_system_prompt += f"\n\n#Summary:\n{summary}\n\nLinkedIn:\n{linkedin}"
evaluation_system_prompt += "With this context, evaluate the latest response whether the response is acceptable and your feedback"

In [10]:
print(evaluation_system_prompt)

You are an evaluator that decides the validity of a response to a question. You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable. The Agent is playing a role of Debmalya Mondal's digital avatar and is representing Debmalya Mondal on their website. The Agent is instructed to be professional and engaging since it will be chatting to a potential client or a future employer. The Agent is also instructed to answer only about Debmalya Mondal's career, experience and skills and humbly refuse to answer any other query. The Agent has been provided with context on Debmalya Mondal in the form of summary and Linkedin details. Here's the information:

#Summary:
I am Debmalya Mondal. I am an IT business analyst, data scientist and AI consultant. I have predominatly worked in capital markets projects. In my 15
years of industry experience, I have played many roles like business analyst, project manager, and back-end

In [11]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the user and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the user: {message}"
    user_prompt += f"Here's the latest response from the Agent: {reply}"
    user_prompt += "Evaluate the latest response whether it is acceptable and share your feedback"
    return user_prompt

In [12]:
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
gemini_api_key = os.getenv("GEMINI_API_KEY")
print(openai_api_key[:6], gemini_api_key[:6])

sk-pro AIzaSy


In [14]:
openai = OpenAI(api_key=openai_api_key)
gemini = OpenAI(api_key=gemini_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

In [15]:
class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

In [19]:
def evaluate(reply, message, history):
    user_prompt = evaluator_user_prompt(reply, message, history)
    messages = [{"role": "system", "content": evaluation_system_prompt}, {"role": "user", "content": user_prompt}]
    evaluation = gemini.chat.completions.parse(model="gemini-2.5-flash", messages=messages, response_format=Evaluation)
    return evaluation.choices[0].message.parsed

In [20]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply = response.choices[0].message.content
    feedback = evaluate(reply, message, history)
    print(feedback)
    return reply

In [21]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


is_acceptable=True feedback="The agent provided a professional and engaging greeting, inviting the user to ask relevant questions about Debmalya Mondal's career, skills, and experience. This aligns perfectly with its instructions."
is_acceptable=False feedback='The agent\'s response is not fully acceptable. While it correctly identifies that "LLM engineering" might not be a direct fit for Debmalya\'s core expertise, it misses an opportunity to mention the "Text Prompt Engineering Techniques Skill Badge" which is directly related to working with Large Language Models. This omission makes the answer less comprehensive than it could be. Additionally, the phrasing "I currently do not have specific information about my experience..." implies the avatar is unaware of Debmalya\'s details, rather than directly stating Debmalya\'s experience (or lack thereof) in that specific area. A better response would acknowledge the prompt engineering skill badge while clarifying the primary areas of exper